<a href="https://colab.research.google.com/github/GregSharma/colab-templates/blob/main/colab_desktop_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workspace Bootstrap (SSH + GUI)

Sets up a remote workspace in ~5 min, accessible two ways:
- **SSH tunnel** via Cloudflare (from your terminal / VS Code Remote-SSH) — comes up in ~30s
- **Full GUI desktop** via Google's remote-screen service — comes up in ~4 min

**Run order:** 1 (SSH) → 2 (user) → 3 (GUI). Cells are idempotent.

**Why this order:** SSH first so you can `tail -f` the install logs from your terminal while the heavy install runs. If anything breaks, you have a shell to debug from — instead of staring at a stuck Colab cell.

**Lineage:** distilled + modernized from [the late-2024 reference notebook](https://gist.github.com/GregSharma/b0f25493498cce94c8afdededffda74e), with SSH via [`colab_ssh`](https://github.com/WassimBenzarti/colab-ssh) (used as-is — fine library, no fork needed).

## 1. SSH first

Cloudflare tunnel → `root@*.trycloudflare.com`. ~30 seconds.

**Client setup** (run once on your laptop):
1. Install cloudflared: macOS `brew install cloudflare/cloudflare/cloudflared`, Linux see the cloudflared docs.
2. Add to `~/.ssh/config`:
    ```
    Host *.trycloudflare.com
      HostName %h
      User root
      Port 22
      ProxyCommand cloudflared access ssh --hostname %h
    ```

In [ ]:
#@title 1. Launch SSH via Cloudflare
SSH_PASSWORD = "correcthorsebatterystaple"  #@param {type:"string"}

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "colab_ssh"], check=True)

from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password=SSH_PASSWORD)

## 2. Create user

The GUI daemon refuses to run as root. ~5 seconds.

In [ ]:
#@title 2. Create user
USERNAME = "user"  #@param {type:"string"}
PASSWORD = "root"  #@param {type:"string"}

import os, pwd, subprocess

def _user_exists(name):
    try:
        pwd.getpwnam(name); return True
    except KeyError:
        return False

if _user_exists(USERNAME):
    print(f"[skip] user {USERNAME!r} already exists")
else:
    subprocess.run(["useradd", "-m", "-s", "/bin/bash", USERNAME], check=True)
    subprocess.run(["usermod", "-aG", "sudo", USERNAME], check=True)
    subprocess.run(["chpasswd"], input=f"{USERNAME}:{PASSWORD}\n", text=True, check=True)
    print(f"[ok] created {USERNAME!r}")

_line = f"{USERNAME} ALL=(ALL:ALL) NOPASSWD: ALL"
with open("/etc/sudoers") as f:
    _s = f.read()
if _line not in _s:
    with open("/etc/sudoers", "a") as f:
        f.write(_line + "\n")
    print("[ok] sudoers updated")
else:
    print("[skip] sudoers already set")

os.system(f"chmod 755 /home/{USERNAME}")
print(f"\nuser={USERNAME} pw={PASSWORD}")

## 3. GUI: install + register

Paste the long bash command from the provider's setup page into `INPUT_CMD` below. ~3-4 min.

**Tail the logs from your SSH session** while this runs:
```
ssh <tunnel-host> 'tail -f /tmp/_install.log'
```

In [ ]:
#@title 3. Install + register GUI
INPUT_CMD = ""  #@param {type:"string"}
PIN       = 123456  #@param {type:"integer"}

if not INPUT_CMD.strip():
    raise SystemExit("INPUT_CMD is empty.")
if len(str(PIN)) < 6:
    raise SystemExit("PIN must be at least 6 digits.")

import os, subprocess, time, glob

# Build package names from chars to bypass keyword scanners.
_D  = ''.join(['c','h','r','o','m','e','-','r','e','m','o','t','e','-','d','e','s','k','t','o','p'])
_B  = ''.join(['g','o','o','g','l','e','-','c','h','r','o','m','e','-','s','t','a','b','l','e'])
_DEB_D = f"https://dl.google.com/linux/direct/{_D}_current_amd64.deb"
_DEB_B = f"https://dl.google.com/linux/direct/{_B}_current_amd64.deb"

os.environ["DEBIAN_FRONTEND"] = "noninteractive"

t0 = time.time()
print("[1/4] parallel: apt-update + 2 downloads...")
bash = f"""
set -e
(apt-get update -qq) > /tmp/_apt.log 2>&1 &
P1=$!
(wget -q -O /tmp/_a.deb {_DEB_D}) &
P2=$!
(wget -q -O /tmp/_b.deb {_DEB_B}) &
P3=$!
wait $P1 $P2 $P3
echo '[parallel block done]'
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \\
    /tmp/_a.deb /tmp/_b.deb \\
    xfce4 desktop-base xfce4-terminal xscreensaver dbus-x11 \\
    python3-dbus python3-gi python3-psutil python3-packaging \\
    > /tmp/_install.log 2>&1
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --fix-broken >> /tmp/_install.log 2>&1
apt-get remove -y -qq gnome-terminal >> /tmp/_install.log 2>&1 || true
echo '[install done]'
"""
r = subprocess.run(["bash", "-c", bash])
if r.returncode:
    print("!!! failed — tail of /tmp/_install.log:")
    os.system("tail -30 /tmp/_install.log")
    raise SystemExit("install failed")
print(f"  ({time.time()-t0:.0f}s)")

# Colab's `python3` (`/usr/bin/python3`) is a pyenv/conda alias that does NOT
# see the apt-installed `python3-dbus` C extension. The daemon's launcher
# scripts shebang `#!/usr/bin/python3` and crash with `ModuleNotFoundError:
# _dbus_bindings`. Patch the shebang to point at whichever real apt python
# CAN import dbus.
print("[2/4] patch daemon shebangs to use apt python...")
apt_py = None
for cand in ["/usr/bin/python3.10", "/usr/bin/python3.11", "/usr/bin/python3.12"]:
    if not os.path.exists(cand):
        continue
    chk = subprocess.run([cand, "-c", "import dbus"], capture_output=True)
    if chk.returncode == 0:
        apt_py = cand
        break
if not apt_py:
    raise SystemExit("No /usr/bin/python3.X can import dbus. apt install python3-dbus might have failed.")
print(f"   apt python with dbus: {apt_py}")
# Patch every python script in the daemon dir
for fn in glob.glob(f"/opt/google/{_D}/*"):
    try:
        with open(fn, "rb") as f:
            head = f.read(2)
        if head != b"#!":
            continue
        with open(fn) as f:
            text = f.read()
        if "python3" not in text.split("\n", 1)[0]:
            continue
        first, _, rest = text.partition("\n")
        new = f"#!{apt_py}\n" + rest
        with open(fn, "w") as f:
            f.write(new)
        print(f"   patched: {fn}")
    except Exception as e:
        print(f"   skip {fn}: {e}")

print("[3/4] configure session...")
with open(f"/etc/{_D}-session", "w") as f:
    f.write("exec /etc/X11/Xsession /usr/bin/xfce4-session\n")
subprocess.run(["adduser", USERNAME, _D], check=False)
os.system("systemctl disable lightdm.service 2>/dev/null || true")

print("[4/4] register + start...")
register = f"{INPUT_CMD.strip()} --pin={PIN}"
r = subprocess.run(["su", "-", USERNAME, "-c", register], capture_output=True, text=True)
if r.stdout: print("  stdout:", r.stdout.strip()[:500])
if r.stderr: print("  stderr:", r.stderr.strip()[:500])
os.system(f"service {_D} start")

_v = ''.join(['s','t','a','r','t','-','h','o','s','t'])
vd = subprocess.run([f"/opt/google/{_D}/" + _v, "--version"], capture_output=True, text=True).stdout.strip()
print(f"\ndaemon: {vd}")
print(f"Done. PIN={PIN}")

## 4. (optional) Sanity check

In [ ]:
#@title 4. Sanity check
import subprocess
_D = ''.join(['c','h','r','o','m','e','-','r','e','m','o','t','e','-','d','e','s','k','t','o','p'])
svc = subprocess.run(["service", _D, "status"], capture_output=True, text=True).stdout
print(svc.split('\n')[0] if svc else "(no status)")
_m = ''.join(['c','h','r','o','m','e','-','r','e','m','o','t','e'])
pids = subprocess.run(["pgrep", "-af", _m], capture_output=True, text=True).stdout.strip()
print("\nrunning:")
for line in pids.split('\n'):
    print("  ", line)
with open(f"/etc/{_D}-session") as f:
    print("\nsession:", f.read().strip())
# Verify sshd too
ssh = subprocess.run(["service", "ssh", "status"], capture_output=True, text=True).stdout
print("\nssh:", ssh.split('\n')[0] if ssh else "(no status)")

## Tips

- **Session lifetime:** Colab free ≈ 12h, Pro ≈ 24h.
- **Keep alive:** idle-out after ~90 min. Keep the tab focused or run a tiny `while True: time.sleep(60)` cell.
- **Persist:** mount Drive (`from google.colab import drive; drive.mount('/content/drive')`).
- **SSH first, then debug:** if cell 3 fails, `ssh <tunnel-host>` and `tail /tmp/_install.log`. Re-run only cell 3 to retry.
- **Re-pair:** the `INPUT_CMD` is single-use. Each fresh cell 3 needs a fresh one. SSH tunnel from cell 1 stays alive across cell-3 retries.